# Imports

In [ ]:
!pip install torch_geometric

In [10]:
import os
import numpy as np
import pandas as pd
import copy
import pprint
import math
import gc

import torch
import torch.nn as nn

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader

from datetime import datetime, timezone, timedelta

from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, confusion_matrix, precision_recall_curve
from scipy.special import expit # It is the optimized sigmoid of Scipy

import random
import json

In [3]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


# Experiment Manager

In [4]:
class ExperimentManager:
    def __init__(self, log_file="./logs/experiments_log.csv", model_dir="./saved_models"):
        self.log_file = log_file
        self.model_dir = model_dir
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

    def log_experiment(self, model_config=None, model_name=None, params=None, metrics=None, model_object=None):
        """
        Record an experiment in CSV format and optionally save the model.

        model_config: Recommended dict:
        - model_name (str)
        - type (str)
        - model_params (dict) # ONLY model hyperparameters
        - prob_threshold (float) # Decision threshold for probabilities (for metrics computation)
        - data_params (dict) # Optional
        - extra_params (dict) # Optional

        model_name: Name of the model (str)

        params: (legacy) Extra dict (for compatibility)
        metrics: Dict with results
        model_object: Model object (for saving)
        """

        if metrics is None:
            metrics = {}
        if params is None:
            params = {}

        # Timezone Argentina
        tz = timezone(timedelta(hours=-3))
        now = datetime.now(tz)

        # Input
        if model_config is not None:
            mname = model_config.get("model_name", model_name)
            mtype = model_config.get("type", None)
            model_params = model_config.get("model_params", {})
            threshold = model_config.get("prob_threshold", None)
            data_params = model_config.get("data_params", {})
            extra_params = model_config.get("extra_params", {})
        else:
            # legacy mode
            mname = model_name
            mtype = params.get("type", None)
            threshold = params.get("prob_threshold", None)
            model_params = params
            data_params = {}
            extra_params = {}

        # Create record
        entry = {
            "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
            "model_name": mname,
        }
        if mtype is not None:
            entry["type"] = mtype
        if threshold is not None:
            entry["prob_threshold"] = threshold

        # Prefijos para evitar colisiones de nombres
        entry.update({f"hp_{k}": v for k, v in (model_params or {}).items()})
        entry.update({f"data_{k}": v for k, v in (data_params or {}).items()})
        entry.update({f"extra_{k}": v for k, v in {**extra_params, **params}.items()
                      if k not in ("type", "prob_threshold")})

        entry.update(metrics)

        # Save in csv
        df_new = pd.DataFrame([entry])
        if os.path.exists(self.log_file):
            df_new.to_csv(self.log_file, mode="a", header=False, index=False)
        else:
            df_new.to_csv(self.log_file, mode="w", header=True, index=False)

        print(f"Experiment recorded in {self.log_file}")

        # Save model
        if model_object is not None:
            metric_key = "AUC-PR" if "AUC-PR" in metrics else ("F1" if "F1" in metrics else None)
            metric_val = metrics.get(metric_key, 0) if metric_key else 0
            safe_key = metric_key or "metric"
            filename = f"{mname}_{now.strftime('%Y%m%d_%H%M')}_{safe_key}_{float(metric_val):.4f}"

            if "sklearn" in str(type(model_object)):
                import joblib
                joblib.dump(model_object, os.path.join(self.model_dir, f"{filename}.joblib"))
            else:
                import torch
                torch.save(model_object.state_dict(), os.path.join(self.model_dir, f"{filename}.pth"))

            print(f"Saved model: {filename}")


# Global instance
#exp_manager = ExperimentManager()

# Load data

In [5]:
class NF_IDS_Dataset(Dataset):
    def __init__(self, root_dir, split='train'):
        # root_dir: (eg: "./dataset_processed")
        # split: 'train', 'val' or 'test'
        self.split_dir = os.path.join(root_dir, split)
        super().__init__(self.split_dir)

        # List files ordered numerically to respect the time
        self.files = sorted(
            [f for f in os.listdir(self.split_dir) if f.endswith('.pt')],
            key=lambda x: int(x.split('_')[1].split('.')[0])
        )

    def len(self):
        return len(self.files)

    def get(self, idx):
        data = torch.load(
            os.path.join(self.split_dir, self.files[idx]),
            weights_only=False # to allow loading complex graph objects
        )
        return data

# Model

In [6]:
class SimpleMLP(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2, output_bias_init=None):
        super().__init__()

        # Input: Only flow attributes (edge_attr)
        # The MLP only needs to know how many features the edge has.
        self.input_dim = edge_dim

        # Note: node_dim is received for compatibility but is not used

        # We use LayerNorm instead of BatchNorm1d
        # LayerNorm does not fail with batch_size=1
        self.net = nn.Sequential(
            nn.Linear(self.input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, 1)
        )

        if output_bias_init is not None:
            self.net[-1].bias.data.fill_(output_bias_init)

    def forward(self, x, edge_index, edge_attr):
        """
        Arguments:
            x, edge_index: These are received to maintain compatibility with
                           train_epoch, but this model ignores them (it doesn't
                           use a graph structure).
            edge_attr: The flow's features (bytes, duration, etc.)
        """
        # We ignore everything except the attributes of the flow
        return self.net(edge_attr)


# Functions


## Metrics

In [7]:
def calculate_metrics_gnn(y_true, y_pred_logits, prob_threshold=0.5):
    """
    y_true: List or array of actual labels (0 or 1).
    y_pred_logits: List or array of raw model outputs (before sigmoid).
    """
    y_true = np.array(y_true)
    logits = np.array(y_pred_logits)

    # Convert logits to probabilities, and to classes (0 or 1) depend on prob_threshold
    probs = expit(logits)
    preds = (probs > prob_threshold).astype(int)

    # Threshold-dependent metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    f2 = fbeta_score(y_true, preds, beta=2, zero_division=0)

    # Threshold-independent metrics
    try:
        ap = average_precision_score(y_true, probs)
        roc = roc_auc_score(y_true, probs)
    except ValueError:
        # This occurs if y_true has only one class (e.g., only benign in the batch)
        ap = 0.0
        roc = 0.5

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2, "AUC-PR": ap, "AUC-ROC": roc,
        "FPR": fpr, "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn), "Total_Flows": len(y_true)
    }



## Train and eval

In [8]:
def train_epoch(model, loader, optimizer, criterion, device, is_temporal=False, batch_steps=10):
    model.train()
    total_loss = 0
    steps = 0  # Count of valid graphs processed (not empty)

    # Loss accumulator for Truncated Backprop
    batch_loss = 0

    # Iterate over the Loader
    # batch_idx helps to know if we are on the last element
    for batch_idx, data in enumerate(loader):
        data = data.to(device)

        # If it is an empty graph, skip
        if data.x.shape[0] == 0:
            continue

        # Forward
        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        # Loss calculation
        loss = criterion(out.view(-1), data.y)
        batch_loss += loss
        steps += 1

        # Backpropagation:
        # 1. Each 'batch_steps' valid steps (e.g., every 10 graphs)
        # 2. Or if it is in the LAST batch of the loader (so as not to lose the remainder)
        is_last_batch = (batch_idx == len(loader) - 1)

        if (steps > 0 and steps % batch_steps == 0) or is_last_batch:
            # Only update if there's something accumulated
            if batch_loss > 0:
                optimizer.zero_grad()
                batch_loss.backward()
                optimizer.step()

                # Reset
                total_loss += batch_loss.item()
                batch_loss = 0

                # IMPORTANT for ST-GNN: Detach memory
                if is_temporal:
                    model.detach_all_memory()

    # Average loss per valid step
    return total_loss / steps if steps > 0 else 0

@torch.no_grad()
def evaluate(model, loader, criterion, device, prob_threshold, is_temporal=False):
    model.eval()
    all_logits = []
    all_trues = []
    total_loss = 0
    steps = 0

    # Clear the memory before starting the evaluation (only if it's temporal)
    if is_temporal:
        model.node_memory = {}

    # Don't need enumerate here because we're not doing backprop
    for data in loader:
        data = data.to(device)

        if data.x.shape[0] == 0: continue

        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        loss = criterion(out.view(-1), data.y)
        total_loss += loss.item()

        all_logits.extend(out.view(-1).cpu().numpy())
        all_trues.extend(data.y.cpu().numpy())
        steps += 1

    metrics = calculate_metrics_gnn(all_trues, all_logits, prob_threshold)
    metrics["Loss"] = total_loss / steps if steps > 0 else 0
    return metrics

## Optimal threshold

In [11]:
def find_optimal_threshold(model, loader, device, is_temporal=False):
    model.eval()

    if is_temporal:
        # Clear the memory before starting the evaluation (only if it's temporal)
        model.node_memory = {}

    all_probs = []
    all_trues = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            if is_temporal:
                out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
            else:
                out = model(data.x, data.edge_index, data.edge_attr)

            # Convert Logits to Probabilities (Sigmoid)
            probs = torch.sigmoid(out.view(-1))

            all_probs.extend(probs.cpu().numpy())
            all_trues.extend(data.y.cpu().numpy())

    all_trues = np.array(all_trues)
    all_probs = np.array(all_probs)

    # 1. Precision-Recall Curve
    precisions, recalls, thresholds = precision_recall_curve(all_trues, all_probs)

    # 2. Calculate F1 for each possible threshold
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    f1_scores = f1_scores[:-1]

    # 3. Find max
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    best_rec = recalls[best_idx]
    best_prec = precisions[best_idx]

    print(f"\nBEST THRESHOLD FOUND: {best_threshold:.4f}")
    print(f"   F1 Score : {best_f1:.4f}")
    print(f"   Recall   : {best_rec:.4f}")
    print(f"   Precision: {best_prec:.4f}")

    # Probability Diagnosis
    avg_benign = np.mean(all_probs[all_trues == 0])
    avg_attack = np.mean(all_probs[all_trues == 1])
    print(f"\nProbability Diagnosis:")
    print(f"   Average assigned to Benignos: {avg_benign:.4f}")
    print(f"   Average assigned to Attacks : {avg_attack:.4f}")

    return best_threshold

## Run multiple seeds

In [12]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

In [13]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)

In [14]:
def run_multiple_seeds(model_class, model_config, train_loader, val_loader,
                       manager,
                       seeds=[42, 100, 2024, 777, 99],
                       epochs=60,
                       device='cpu',
                       experiment_name="Experiment"):

    results_summary = {
        'seeds': [],
        'best_aucpr': [],
        'best_f1': [],
        'best_threshold': []
    }

    all_losses = {}

    print(f" Starting Multi-Seed Run: {experiment_name}")
    print(f"   Seeds: {seeds}")
    print("-" * 60)

    for seed in seeds:
        # Name
        exp_id = experiment_name + f"_seed{seed}"
        print(f"\n{exp_id}")

        # Preventive Memory Cleaning (Before loading anything)
        gc.collect()
        torch.cuda.empty_cache()

        # Update model_config
        model_config['model_name'] = exp_id

        print(f"\n Seed: {seed} ...")
        set_seed(seed)

        # 1. Instantiate Model
        model = model_class(**model_config['model_params']).to(device)

        # 2. Training Setup
        optimizer = torch.optim.Adam(model.parameters(), lr=model_config['extra_params']['learning_rate'])
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([model_config['extra_params']['pos_weight']]).to(device))

        # Variables for tracking within this seed
        train_loss_history = []
        val_loss_history = []

        best_auc_seed = 0
        best_metrics_seed = {}
        best_model_wts = copy.deepcopy(model.state_dict())

        # Epochs
        for epoch in range(1, epochs + 1):
            # Detect whether it is a temporal model (ST or GRU) to adjust inputs
            is_temporal = 'GRU' in experiment_name or 'ST' in experiment_name

            # --- TRAIN ---
            loss_train = train_epoch(model, train_loader, optimizer, criterion,
                                     device=device, is_temporal=is_temporal,
                                     batch_steps=model_config['extra_params']["batch_steps"])

            # --- EVAL ---
            metrics = evaluate(model, val_loader, criterion, device,
                               prob_threshold=model_config['prob_threshold'], is_temporal=is_temporal)

            # Save history
            train_loss_history.append(float(loss_train))
            val_loss_history.append(float(metrics['Loss']))

            if (epoch+1) % 10 == 0: # Print every 10 epochs
                print(f"   Ep {epoch+1} | Loss: {loss_train:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val Rec: {metrics['Recall']:.4f}")

            # 4. Checkpoint of the best model of THIS seed
            if metrics['AUC-PR'] > best_auc_seed:
                best_auc_seed = metrics['AUC-PR']
                best_metrics_seed = metrics
                best_model_wts = copy.deepcopy(model.state_dict())

        # 5. LOGGING WITH EXPERIMENT MANAGER
        model.load_state_dict(best_model_wts)
        manager.log_experiment(
            model_config=model_config,
            metrics=best_metrics_seed,
            model_object=model
        )

        print(f"\nBEST AUC-PR: {best_auc_seed:.4f} (FPR: {metrics['FPR']:.4f})")
        best_threshold_seed = find_optimal_threshold(model, val_loader, device, is_temporal)

        # 6. Accumulate for JSON report
        results_summary['seeds'].append(seed)
        results_summary['best_aucpr'].append(float(best_auc_seed))
        results_summary['best_f1'].append(float(best_metrics_seed.get('F1', 0)))
        results_summary['best_threshold'].append(float(best_threshold_seed))

        all_losses[f"seed_{seed}"] = {
            'train': train_loss_history,
            'val': val_loss_history
        }

        # 7. Save JSON
        filename_json = f"./logs/curves_multiseed_{experiment_name}.json"
        try:
            with open(filename_json, 'w') as f:
                json.dump({'summary': results_summary, 'losses': all_losses}, f, cls=NumpyEncoder)
            print(f"\nProgressive backup of losses saved in: {filename_json}")
        except Exception as e:
            print(f"\nWarning: JSON backup could not be saved: {e}")

        print(f"\n End seed {seed}")

        print("-" * 60)

    # 8. Print final statistics
    avg_auc = np.mean(results_summary['best_aucpr'])
    std_auc = np.std(results_summary['best_aucpr'])

    print("\n" + "="*50)
    print(f" AVERAGE RESULT: {experiment_name}")
    print(f"   AUC-PR: {avg_auc:.4f} ± {std_auc:.4f}")
    print("="*50)


# Auxiliary

In [15]:
ROOT_PATH = "./dataset_processed"

In [16]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)

Train size: 1998 | Val size: 428


# Main

## First configuration

In [17]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_mlp_biasinit.csv")

In [18]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 10
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.001

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 64
DROPOUT = 0.2

PROB_THRESHOLD = 0.5


print(f"Using device: {DEVICE}")


Using device: cpu


In [ ]:
model_config = {
    "model_name": "EdgeMLP",
    "type": "MLP",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": bias_value,
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": 1.0,
        "learning_rate": LR
    }
}


In [ ]:
mlp_model = EdgeMLP(**model_config['model_params']).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value]).to(DEVICE))
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=LR)


best_aucpr = 0.0
best_wts = copy.deepcopy(mlp_model.state_dict())
best_metrics = {}

print(f"--- Starting training: {model_config['model_params']} ---")

for epoch in range(EPOCHS):
    # Note: batch_steps here acts as a simple "gradient accumulation"
    loss = train_epoch(mlp_model, train_loader, optimizer, criterion, DEVICE)
    metrics = evaluate(mlp_model, val_loader, criterion, DEVICE, prob_threshold=0.5)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val F1: {metrics['F1']:.4f} | Val Rec: {metrics['Recall']:.4f}")

    if metrics['AUC-PR'] > best_aucpr:
        best_aucpr = metrics['AUC-PR']
        best_metrics = metrics
        best_wts = copy.deepcopy(mlp_model.state_dict())


print(f"\nRestoring better weights (AUC-PR: {best_aucpr:.4f})...")
mlp_model.load_state_dict(best_wts)
exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=mlp_model)

print(f"\nBest AUC-PR for EdgeMLP obtained: {best_aucpr:.4f}")

## Search grid

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 10
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 64
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5


search_space = {
    'pos_weight': [1.0, 2.0, 5.0],
    'hidden_dim': [32, 64]
}


In [ ]:
model_config = {
    "model_name": None,
    "type": "MLP",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": None,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE,
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": None,
        "learning_rate": LR
    }
}

In [ ]:
# Load existing history to avoid repeating
existing_experiments = []
if os.path.exists(exp_manager.log_file):
    try:
        df_log = pd.read_csv(exp_manager.log_file)
        if 'model_name' in df_log.columns:
            existing_experiments = df_log['model_name'].values.tolist()
        print(f"History loaded. {len(existing_experiments)} experiments already performed")
    except:
        print("The current log could not be read, we will start from scratch")

count = 0
total_exps = len(search_space['pos_weight']) * len(search_space['hidden_dim'])

for pw in search_space['pos_weight']:
    for h_dim in search_space['hidden_dim']:
        count += 1

        # Name
        exp_id = f"EdgeMLP_W{int(pw)}_H{h_dim}"
        print(f"\n{exp_id}")

        if exp_id in existing_experiments:
            print(f"[{count}/{total_exps}] Skipping {exp_id} (Already registered in CSV)")
            continue

        print(f"\n[{count}/{total_exps}] Starting: {exp_id}")

        # Preventive Memory Cleaning (Before loading anything)
        gc.collect()
        torch.cuda.empty_cache()

        # Update model_config
        model_config['model_name'] = exp_id
        model_config["model_params"]["hidden_dim"] = h_dim
        model_config["extra_params"]["pos_weight"] = pw

        try:
            # Instantiate
            model = EdgeMLP(**model_config['model_params']).to(DEVICE)

            # Configure training
            optimizer = torch.optim.Adam(model.parameters(), lr=LR)
            criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw]).to(DEVICE))

            # Deepcopy vars
            best_aucpr = 0.0
            best_wts = copy.deepcopy(model.state_dict())
            best_metrics = {}

            for epoch in range(EPOCHS):
                loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE, is_temporal=False, batch_steps=BATCH_STEPS)
                metrics = evaluate(model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=False)

                print(f"   Ep {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val Rec: {metrics['Recall']:.4f}")

                if metrics['AUC-PR'] > best_aucpr:
                    best_aucpr = metrics['AUC-PR']
                    best_metrics = metrics
                    best_wts = copy.deepcopy(model.state_dict())

            # Restore and save
            model.load_state_dict(best_wts)
            exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=model)
            print(f"   Best AUC-PR: {best_aucpr:.4f} (FPR: {metrics['FPR']:.4f})")

        except Exception as e:
            print(f"   FATAL ERROR in {exp_id}: {str(e)}")
            continue

        # Clean memory
        del model
        del optimizer
        del criterion
        del best_wts
        gc.collect()
        torch.cuda.empty_cache()


## Seeds

In [19]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5



Using device: cpu


In [24]:
model_config = {
    "model_name": None,
    "type": "SimpleMLP",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE,
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [21]:
exp_manager = ExperimentManager(log_file="./logs/simpleMLP_biasinit.csv", model_dir="./saved_models_simpleMLP")

In [23]:
run_multiple_seeds(
    model_class=SimpleMLP,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name="SimpleMLP_BiasOn"
)

 Starting Multi-Seed Run: SimpleMLP_BiasOn
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

SimpleMLP_BiasOn_seed42

 Seed: 42 ...
   Ep 10 | Loss: 0.1938 | Val AUC-PR: 0.1730 | Val Rec: 0.0000
   Ep 20 | Loss: 0.1878 | Val AUC-PR: 0.1733 | Val Rec: 0.0000
   Ep 30 | Loss: 0.1862 | Val AUC-PR: 0.1797 | Val Rec: 0.0000
   Ep 40 | Loss: 0.1824 | Val AUC-PR: 0.1941 | Val Rec: 0.0000
   Ep 50 | Loss: 0.1785 | Val AUC-PR: 0.2114 | Val Rec: 0.1765
   Ep 60 | Loss: 0.1756 | Val AUC-PR: 0.2326 | Val Rec: 0.1852
Experiment recorded in ./logs/simpleMLP_biasinit.csv
Saved model: SimpleMLP_BiasOn_seed42_20260212_1039_AUC-PR_0.2326

BEST AUC-PR: 0.2326 (FPR: 0.0054)

BEST THRESHOLD FOUND: 0.2080
   F1 Score : 0.2926
   Recall   : 0.1979
   Precision: 0.5605

Probability Diagnosis:
   Average assigned to Benignos: 0.0768
   Average assigned to Attacks : 0.2704

Progressive backup of losses saved in: ./logs/curves_multiseed_SimpleMLP_BiasOn.json

 End s